# Naver 뉴스 URL 수집 (Colab용)
- Google Drive에 결과 저장
- 날짜별 루프로 2000건 제한 우회
- 중간에 끊겨도 임시 파일 기반으로 이어서 수집 가능

In [1]:
# Colab 환경 세팅 — Selenium, Google Chrome 설치
!wget -q -O /tmp/google-chrome.deb https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get install -y -q /tmp/google-chrome.deb
!pip install -q selenium


Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  at-spi2-core gsettings-desktop-schemas libatk-bridge2.0-0 libatk1.0-0
  libatk1.0-data libatspi2.0-0 libvulkan1 libxcomposite1 libxtst6
  mesa-vulkan-drivers session-migration
The following NEW packages will be installed:
  at-spi2-core google-chrome-stable gsettings-desktop-schemas
  libatk-bridge2.0-0 libatk1.0-0 libatk1.0-data libatspi2.0-0 libvulkan1
  libxcomposite1 libxtst6 mesa-vulkan-drivers session-migration
0 upgraded, 12 newly installed, 0 to remove and 3 not upgraded.
Need to get 11.2 MB/141 MB of archives.
After this operation, 478 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 libatk1.0-data all 2.36.0-3build1 [2,824 B]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libatk1.0-0 amd64 2.36.0-3build1 [51.9 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/main amd64 libatspi2.0-0 a

In [2]:
# Google Drive 마운트 — 중간에 끊겨도 데이터 보존
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [9]:
# Drive 안의 프로젝트 폴더로 이동
import os
PROJECT_DIR = '/content/drive/MyDrive/Text-data-Analysis_26-Spring'
os.chdir(PROJECT_DIR)
print(f'현재 작업 폴더: {os.getcwd()}')


현재 작업 폴더: /content/drive/MyDrive/Text-data-Analysis_26-Spring


In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
import time
from datetime import datetime, timedelta
import urllib.parse
import json
import os
import shutil
import subprocess
import calendar

# 로컬/Colab 비교를 위해 User-Agent 고정
USER_AGENT = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36'

# 담당 키워드와 수집할 시작/끝 년월 지정
# query 하나당 start_ym부터 end_ym까지 월 단위 작업으로 자동 분할, start_ym/end_ym 형식: 'YYYY.MM'
query_ranges = [
    {'query': 'SKT', 'start_ym': '2025.04', 'end_ym': '2025.08'},
    # {'query': 'KT', 'start_ym': '2025.09', 'end_ym': '2026.01'},
    # {'query': 'LG U+', 'start_ym': '2025.08', 'end_ym': '2025.12'},
]


# 월 단위 작업 목록 생성
# 월 마지막 일자는 calendar.monthrange로 자동 계산하므로 28/29/30/31일을 직접 입력하지 않아도 됨
def build_monthly_jobs(query_ranges):
    jobs = []

    for item in query_ranges:
        query = item['query']
        start_year, start_month = map(int, item['start_ym'].split('.'))
        end_year, end_month = map(int, item['end_ym'].split('.'))

        # 시작 년월이 끝 년월보다 늦으면 작업 범위가 잘못된 것이므로 즉시 중단
        if (start_year, start_month) > (end_year, end_month):
            raise ValueError(f"시작 년월이 끝 년월보다 늦습니다: {item}")

        # 시작 월부터 끝 월까지 한 달씩 이동하면서 작업 생성
        year, month = start_year, start_month
        while (year, month) <= (end_year, end_month):
            # 해당 월의 마지막 날짜 자동 계산 (윤년 2월 29일 포함)
            last_day = calendar.monthrange(year, month)[1]
            jobs.append({
                'query': query,
                'start_date': f'{year}.{month:02d}.01',
                'end_date': f'{year}.{month:02d}.{last_day:02d}',
            })

            # 다음 달로 이동, 12월 다음은 다음 해 1월로 변경
            month += 1
            if month == 13:
                year += 1
                month = 1

    return jobs


# 변수 정의 (날짜 형식: 'YYYY.MM.DD')
# 생성된 jobs는 셀 5에서 순서대로 실행
jobs = build_monthly_jobs(query_ranges)

print(f'총 작업 수: {len(jobs)}')
for job in jobs:
    print(job)

# 셀 3을 건너뛰고 실행해도 기본 프로젝트 경로를 사용할 수 있게 보완
try:
    PROJECT_DIR
except NameError:
    PROJECT_DIR = '/content/drive/MyDrive/Text-data-Analysis_26-Spring'

# 저장할 폴더 지정
# 링크 파일, 임시 파일, 수집 로그, 실패 목록이 모두 이 폴더에 저장
SAVE_DIR = os.path.join(PROJECT_DIR, 'notebook', 'crawling', 'data')
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'저장 위치: {SAVE_DIR}')

options = Options()
options.add_argument(f'user-agent={USER_AGENT}')  # 요청 환경을 일정하게 유지하기 위해 User-Agent 고정
print(f'User-Agent: {USER_AGENT}')
options.add_experimental_option('excludeSwitches', ['enable-automation'])  # 자동화 제어 관련 switch 제외
options.add_experimental_option('useAutomationExtension', False)  # Selenium 자동화 확장 비활성화
options.add_argument('--disable-blink-features=AutomationControlled')  # AutomationControlled 플래그 비활성화
options.add_argument('--headless=new')  # Colab은 GUI가 없으므로 새 headless 모드 사용
options.add_argument('--no-sandbox')  # Colab 컨테이너 환경에서 Chrome 실행 안정화
options.add_argument('--disable-dev-shm-usage')  # /dev/shm 용량 부족으로 Chrome이 죽는 문제 완화
options.add_argument('--disable-gpu')  # headless 환경에서 GPU 관련 오류 방지
options.add_argument('--window-size=1920,1080')  # headless에서도 일정한 화면 크기로 렌더링

# Colab chromium-browser 패키지는 snap 래퍼라 Selenium에서 자주 실패
# 설치 셀에서 받은 Google Chrome 사용, ChromeDriver는 Selenium Manager에 맡김
chrome_binary = shutil.which('google-chrome') or shutil.which('google-chrome-stable') or '/usr/bin/google-chrome'
if not os.path.exists(chrome_binary):
    raise FileNotFoundError('Google Chrome을 찾지 못했습니다. 설치 셀을 먼저 다시 실행해 주세요.')

options.binary_location = chrome_binary
print(f'Chrome binary: {chrome_binary}')
subprocess.run([chrome_binary, '--version'], check=False)

# Chrome 드라이버 설정
# Selenium Manager가 현재 Chrome 버전에 맞는 ChromeDriver를 자동으로 찾거나 내려받음
service = Service()
driver = webdriver.Chrome(service=service, options=options)

# navigator.webdriver 플래그 제거
driver.execute_cdp_cmd('Page.addScriptToEvaluateOnNewDocument', {
    'source': 'Object.defineProperty(navigator, "webdriver", {get: () => undefined})'
})

# 창 열기는 아래 셀의 날짜 루프 안에서 날짜별로 자동 실행


In [ ]:
import random

# 서버 부담을 줄이기 위해 날짜/job 사이에 짧은 랜덤 대기
DAY_PAUSE_RANGE_SEC = (2, 5)
JOB_PAUSE_RANGE_SEC = (10, 25)
SKIP_COMPLETED = True

# 랜덤 대기 후 로그 출력
def polite_sleep(label, pause_range):
    pause_sec = random.uniform(*pause_range)
    print(f"{label} {pause_sec:.1f}초 대기")
    time.sleep(pause_sec)

# 날짜를 하루씩 쪼개서 링크 수집 후 임시 파일에 누적 저장
# 이미 임시 파일이 있으면 이어서 누적 (중복은 자동 제거)
def collect_links(query, start_date, end_date, save_dir=SAVE_DIR, driver=driver):
    # temp_links_path: 중간에 끊겼을 때 이어서 수집하기 위한 체크포인트
    # links_save_path: 월별 최종 URL 목록
    # stats_save_path: 날짜별 수집 개수, 스크롤 횟수, 소요 시간 로그
    temp_links_path = os.path.join(save_dir, f'{query}_{start_date}_{end_date}_temp_links.json')
    links_file_name = f"링크_{query}_{start_date.replace('.', '')[2:]}_{end_date.replace('.', '')[2:]}.json"
    links_save_path = os.path.join(save_dir, links_file_name)
    stats_file_name = f"수집로그_{query}_{start_date.replace('.', '')[2:]}_{end_date.replace('.', '')[2:]}.json"
    stats_save_path = os.path.join(save_dir, stats_file_name)

    # 최종 파일이 이미 있으면 같은 월은 건너뜀
    # 밤새 실행 중 재시작해도 이미 완료된 월을 다시 긁지 않기 위한 장치
    if SKIP_COMPLETED and os.path.exists(links_save_path):
        print()
        print(f"=== {query} / {start_date} ~ {end_date} 이미 완료됨, 건너뜀 ===")
        print(f"기존 파일: {links_save_path}")
        return links_save_path

    print()
    print(f"=== {query} / {start_date} ~ {end_date} 수집 시작 ===")

    # 임시 파일에 기존 링크가 있으면 불러오기
    # 다음 실행에서는 last_date 다음 날짜부터 이어서 수집
    if os.path.exists(temp_links_path):
        with open(temp_links_path, 'r', encoding='utf-8') as f:
            checkpoint = json.load(f)
        # set으로 변환해서 이미 모은 링크와 새 링크의 중복 제거
        all_links_set = set(checkpoint.get('links', []))
        last_collected_date = checkpoint.get('last_date')
        # daily_stats도 임시 파일에 같이 보관해서 중단 전 로그 유지
        daily_stats = checkpoint.get('daily_stats', [])
        print(f"기존 임시 파일에서 링크 {len(all_links_set)}개 불러옴 — {last_collected_date} 다음부터 이어서 수집")
    else:
        all_links_set = set()
        last_collected_date = None
        daily_stats = []
        print('새로 링크 수집 시작')

    # start_date ~ end_date를 하루씩 순회
    current = datetime.strptime(start_date, '%Y.%m.%d')
    end = datetime.strptime(end_date, '%Y.%m.%d')

    while current <= end:
        day_str = current.strftime('%Y.%m.%d')

        # 이미 수집 완료한 날짜면 건너뜀
        if last_collected_date and day_str <= last_collected_date:
            print(f"{day_str} — 이미 수집 완료, 건너뜀")
            current += timedelta(days=1)
            continue

        # 하루짜리 검색 URL 생성
        # 네이버 뉴스 검색은 결과가 많을 때 전체 기간으로 검색하면 제한이 생길 수 있어 하루 단위로 쪼갬
        # 주소 원문 해당 구간 : so%3Ar%2Cp%3Afrom20220816to20220831 -> ':', ',' 처리를 위해 인코딩 필요
        encoded_query = urllib.parse.quote(query)
        nso_value = f"so:r,p:from{day_str.replace('.', '')}to{day_str.replace('.', '')}"
        url = (
            f'https://search.naver.com/search.naver?ssc=tab.news.all'
            f'&query={encoded_query}&sm=tab_opt&sort=2&photo=0&field=0'
            f'&pd=3&ds={day_str}&de={day_str}&docid=&related=0&mynews=0'
            f'&office_type=0&office_section_code=0&news_office_checked='
            f'&nso={urllib.parse.quote(nso_value)}&is_sug_officeid=0'
            f'&office_category=&service_area=2'
        )

        started_at = time.time()
        driver.get(url)

        # 페이지 끝까지 내리기
        # 단, 한 번 멈춘 것처럼 보여도 늦게 로딩될 수 있어 2초 더 기다린 뒤 재확인
        last_height = driver.execute_script('return document.body.scrollHeight')
        pause_sec = 1.5
        scroll_count = 0

        while True:
            scroll_count += 1
            # 스크롤 끝까지 내리기
            driver.execute_script('window.scrollTo(0, document.body.scrollHeight);')

            # 스크롤 내린 후 페이지 로딩을 위한 시간이 필요하다면, pause_sec에 숫자 입력
            time.sleep(pause_sec)

            # 스크롤 내린 후의 페이지 높이 = new_height
            new_height = driver.execute_script('return document.body.scrollHeight')

            if new_height == last_height:
                # 네트워크 지연으로 콘텐츠가 늦게 로딩될 수 있으므로 한 번 더 대기 후 재확인
                time.sleep(2.0)
                final_height = driver.execute_script('return document.body.scrollHeight')
                if final_height == new_height:
                    break  # 재확인에도 높이 변화 없으면 진짜 끝
                last_height = final_height  # 추가 콘텐츠가 로딩됐으면 계속 스크롤
                continue

            # 스크롤 내린 후의 페이지 높이(new_height)를 현재 페이지 높이(last_height) 변수에 저장
            last_height = new_height

        # n.news.naver.com을 포함하는 링크 set에 추가 (중복 자동 제거)
        a_tags = driver.find_elements(By.XPATH, '//a[contains(@href, "n.news.naver.com")]')
        day_links = {a.get_attribute('href') for a in a_tags if a.get_attribute('href')}
        before = len(all_links_set)
        all_links_set.update(day_links)
        added = len(all_links_set) - before
        elapsed = round(time.time() - started_at, 2)

        # 날짜별 수집량/스크롤 횟수/소요 시간 로그 저장
        # 나중에 특정 날짜가 유독 적게 수집됐는지 점검할 때 사용
        stat = {
            'date': day_str,
            'found': len(day_links),
            'added': added,
            'total': len(all_links_set),
            'scroll_count': scroll_count,
            'elapsed_sec': elapsed,
        }
        daily_stats.append(stat)

        print(
            f"{day_str} — {len(day_links)}건 수집 / 신규 {added}건 추가 "
            f"/ 누적 {len(all_links_set)}건 / 스크롤 {scroll_count}회 / {elapsed}초"
        )

        # 하루치 수집 후 임시 파일에 즉시 저장 (중간에 끊겨도 누적 보존 + 마지막 완료 날짜 기록)
        with open(temp_links_path, 'w', encoding='utf-8') as f:
            json.dump(
                {'links': sorted(all_links_set), 'last_date': day_str, 'daily_stats': daily_stats},
                f,
                ensure_ascii=False,
                indent=2,
            )
        last_collected_date = day_str
        current += timedelta(days=1)
        polite_sleep('다음 날짜 전', DAY_PAUSE_RANGE_SEC)

    # 월별 링크 최종 저장본 생성
    naver_news_links = sorted(all_links_set)

    with open(links_save_path, 'w', encoding='utf-8') as f:
        json.dump(naver_news_links, f, ensure_ascii=False, indent=2)

    with open(stats_save_path, 'w', encoding='utf-8') as f:
        json.dump(daily_stats, f, ensure_ascii=False, indent=2)

    # 임시 파일 삭제
    if os.path.exists(temp_links_path):
        os.remove(temp_links_path)

    print(f"수집 완료 — 총 {len(naver_news_links)}개")
    print(f"링크 저장: {links_save_path}")
    print(f"수집 로그 저장: {stats_save_path}")
    return links_save_path

# 한 작업이 실패해도 실패 목록에 기록하고 다음 작업으로 넘어감
# 실패 목록은 마지막에 수집실패목록.json으로 저장
results = []
failures = []
for index, job in enumerate(jobs, start=1):
    print()
    print(f"[{index}/{len(jobs)}] 작업 실행: {job}")
    try:
        # job 딕셔너리의 query/start_date/end_date를 collect_links 인자로 전달
        results.append(collect_links(**job))
    except Exception as exc:
        # 한 월에서 오류가 나도 전체 작업이 멈추지 않도록 실패 정보만 저장
        failures.append({'job': job, 'error': repr(exc)})
        print(f"작업 실패, 다음 작업으로 넘어갑니다: {exc!r}")
    finally:
        if index < len(jobs):
            # 다음 월별 job으로 넘어가기 전 대기
            polite_sleep('다음 작업 전', JOB_PAUSE_RANGE_SEC)

# 실패한 작업이 있으면 나중에 다시 돌릴 수 있게 파일로 저장
if failures:
    failures_path = os.path.join(SAVE_DIR, '수집실패목록.json')
    with open(failures_path, 'w', encoding='utf-8') as f:
        json.dump(failures, f, ensure_ascii=False, indent=2)
    print()
    print(f"실패 작업 {len(failures)}개 저장: {failures_path}")

print()
print('전체 작업 완료')
print(f'성공/건너뜀: {len(results)}개, 실패: {len(failures)}개')
for result_path in results:
    print(result_path)


In [15]:
# 브라우저 창 닫기
driver.quit()